In [10]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scipy.ndimage as ndi

def generate_decode_image(decode_xyi, h, w):    
    """
    Generates a 2D image from a storm table by mapping localizations to pixel coordinates.

    Parameters:
        decode_xyi ... TODO
        h (int): Height of the output image in pixels.
        w (int): Width of the output image in pixels.

    Returns:
        numpy.ndarray: A 2D array of shape (h, w) representing the generated image, where each 
                       pixel value corresponds to the summed intensity and offset of localizations 
                       mapped to that pixel.

    Notes:
        - The function assumes a fixed scaling factor (kappa = 1/60 nm/px) to convert 
          nanometer coordinates to pixel coordinates.
        - It works with fixed upscaling factor 4x
        - Localizations are rounded to the nearest pixel using nearest-neighbor rounding.
        - If a localization falls outside the image bounds, it is ignored.
        - If multiple localizations map to the same pixel, their intensities and offsets are summed.
        - A warning message is printed if a pixel is overwritten by multiple localizations.
    """
    #"id","frame","x [nm]","y [nm]","sigma [nm]","intensity [photon]","offset [photon]","bkgstd [photon]","chi2","uncertainty [nm]"
    img = np.zeros((h,w), dtype=np.float64)
    kappa = 1#1/60 #nm/px of object space
    offset_comp = 4
    #read storm table
    for x, y, intensity in decode_xyi:
        #fixed upscale factor 4 here, factor 1.5 is due to grid fitting
        
        xnear = int(np.round(x*4*kappa)+1.5+offset_comp)
        ynear = int(np.round(y*4*kappa)+1.5+offset_comp)
        if (xnear >= w) or (xnear < 0) or (ynear >= h) or (ynear < 0):
            continue
        if img[ynear, xnear] != 0:
            print("Ooops!", xnear, ynear, 'is already there')
        img[ynear, xnear] += (intensity)
    return img

def l1norm(img):
    img = img - np.min(img)
    return img/np.sum(img)


In [11]:

SUFFIXES = ['defocused', 'astigmatism', 'spherical']
INPUTS = [f'data/decode_{key}_star.h5' for key in SUFFIXES]
#'cnn_output_star.h5'
OUTPUTS = [f'data/decode_render_star_{key}.h5' for key in SUFFIXES]

for IN, OUT in zip(INPUTS, OUTPUTS):
    print(f'{IN}->{OUT}')
        
    with h5py.File(IN, 'r') as h5fi:
        print(h5fi['data'].keys())
        xyz = np.array(h5fi['data/xyz'])
        phot = np.array(h5fi['data/phot'])
        fridx = np.array(h5fi['data/frame_ix'])
    
    stack = np.zeros((len(set(fridx)),200,200), dtype=np.float32)
    
    #group emitters together by its frame and render it into grid
    for i, idx in enumerate(sorted(list(set(fridx)))):
        sel = (fridx==idx)
        frame_xyz = xyz[sel]
        frame_int = phot[sel]
        frame_xyi = np.hstack([frame_xyz[:,0:2], frame_int.reshape((-1,1))])    
        im = generate_decode_image(frame_xyi, 200, 200)
        stack[i] = im
    
    with h5py.File(OUT, 'w') as h5fo:
        h5fo.create_dataset('rendered_frames', data = stack)



data/decode_defocused_star.h5->data/decode_render_star_defocused.h5
<KeysViewHDF5 ['bg', 'bg_cr', 'bg_sig', 'frame_ix', 'id', 'phot', 'phot_cr', 'phot_sig', 'prob', 'xyz', 'xyz_cr', 'xyz_sig']>
data/decode_astigmatism_star.h5->data/decode_render_star_astigmatism.h5
<KeysViewHDF5 ['bg', 'bg_cr', 'bg_sig', 'frame_ix', 'id', 'phot', 'phot_cr', 'phot_sig', 'prob', 'xyz', 'xyz_cr', 'xyz_sig']>
data/decode_spherical_star.h5->data/decode_render_star_spherical.h5
<KeysViewHDF5 ['bg', 'bg_cr', 'bg_sig', 'frame_ix', 'id', 'phot', 'phot_cr', 'phot_sig', 'prob', 'xyz', 'xyz_cr', 'xyz_sig']>
